# 📊 Tesis Evaluation: Ground Truth Perbandingan (Compare Systems)
Notebook ini digunakan untuk menghitung dan membandingkan nilai **NDCG@5** antara **Legal Tina** dan **Sistem Vector Search (Qdrant)** secara langsung dari data *snapshot* Ground Truth tanpa melakukan *live query* ke database.

In [20]:
import numpy as np
import pandas as pd
print("✅ Libraries berhasil di-import!")

✅ Libraries berhasil di-import!


### 🛠️ Helper Functions
Fungsi `parse_rank` digunakan untuk memecah string peringkat `[Agreement Rank],[Article Rank]` (contoh: `"1,11"` menjadi tuple `(1, 11)`)

In [15]:
def parse_rank(rank_str):
    """Mengubah format string '1,4' menjadi tuple (1, 4) untuk sorting numerik."""
    if pd.isna(rank_str) or str(rank_str).strip() == "" or str(rank_str).lower() == "nan":
        return None
    try:
        cleaned = str(rank_str).replace('"', '').replace("'", "").strip()
        parts = cleaned.split(',')
        return (int(float(parts[0])), int(float(parts[1])))
    except:
        return None

def compute_dcg(scores):
    """Menghitung nilai Discounted Cumulative Gain (DCG)"""
    return sum(rel / np.log2(idx + 2) for idx, rel in enumerate(scores))

### 🏁 Core Compare Evaluation Function
Fungsi ini akan memisahkan pool data untuk Legal Tina dan Qdrant, mengurutkannya berdasarkan ranking internal masing-masing sistem, lalu membandingkan nilai NDCG keduanya per kueri.

In [18]:
def evaluate_compare_gt(file_path, limit_k=10):
    print(f"📖 Membaca file Ground Truth Compare: {file_path}")
    df = pd.read_csv(file_path)
    
    # Ambil daftar kueri unik
    unique_queries = df[['query_id', 'query']].drop_duplicates()
    
    compare_reports = []
    
    print("\n" + "="*85)
    print(f"🏆 HASIL ADU PERFORMA SISTEM: LEGAL TINA VS VECTOR SEARCH (NDCG@{limit_k})")
    print("="*85)
    
    for _, q_row in unique_queries.iterrows():
        q_id = q_row['query_id']
        q_text = q_row['query']
        
        # Filter data murni milik kueri ini
        df_query = df[df['query_id'] == q_id].copy()
        
        # 1. HITUNG POOL IDEAL (IDCG) BERSAMA PER KUERI
        all_scores = df_query['relevance'].tolist()
        ideal_scores = sorted(all_scores, reverse=True)[:limit_k]
        idcg = compute_dcg(ideal_scores)
        
        # 2. PROSES DATA LEGAL TINA
        df_tina = df_query[df_query['rank_legal_tina'].notna()].copy()
        df_tina['parsed_rank'] = df_tina['rank_legal_tina'].apply(parse_rank)
        df_tina = df_tina.dropna(subset=['parsed_rank']).sort_values(by='parsed_rank')
        tina_scores = df_tina['relevance'].tolist()[:limit_k]
        
        # Genapi dengan 0 jika kurang dari limit_k
        if len(tina_scores) < limit_k:
            tina_scores.extend([0] * (limit_k - len(tina_scores)))
            
        dcg_tina = compute_dcg(tina_scores)
        ndcg_tina = dcg_tina / idcg if idcg > 0 else 0.0
        
        # 3. PROSES DATA VECTOR SEARCH (QDRANT)
        df_qdrant = df_query[df_query['rank_qdrant'].notna()].copy()
        df_qdrant['parsed_rank'] = df_qdrant['rank_qdrant'].apply(parse_rank)
        df_qdrant = df_qdrant.dropna(subset=['parsed_rank']).sort_values(by='parsed_rank')
        qdrant_scores = df_qdrant['relevance'].tolist()[:limit_k]
        
        # Genapi dengan 0 jika kurang dari limit_k
        if len(qdrant_scores) < limit_k:
            qdrant_scores.extend([0] * (limit_k - len(qdrant_scores)))
            
        dcg_qdrant = compute_dcg(qdrant_scores)
        ndcg_qdrant = dcg_qdrant / idcg if idcg > 0 else 0.0
        
        # Tampilkan Log Per Kueri
        print(f"\n▶️ [QUERY {q_id}] '{q_text}'")
        print(f"   Legal Tina    -> Urutan Skor Relevansi: {tina_scores} | NDCG: {ndcg_tina:.4f}")
        print(f"   Vector Search -> Urutan Skor Relevansi: {qdrant_scores} | NDCG: {ndcg_qdrant:.4f}")
        
        compare_reports.append({
            "Query ID": q_id,
            "Query": q_text,
            "NDCG Legal Tina": round(ndcg_tina, 4),
            "NDCG Vector Search": round(ndcg_qdrant, 4),
            "Pemenang": "Vector Search 🚀" if ndcg_qdrant > ndcg_tina else ("Legal Tina 🏛️" if ndcg_tina > ndcg_qdrant else "SERI 🤝")
        })
        
    # 1. Buat DataFrame
    pd.set_option('display.max_columns', None)
    pd.set_option('display.width', 1000)
    pd.set_option('display.colheader_justify', 'center')

    df_report = pd.DataFrame(compare_reports)

    # 2. Hitung Rata-rata
    mean_tina = df_report['NDCG Legal Tina'].mean()
    mean_vector = df_report['NDCG Vector Search'].mean()

    # 3. Cetak Laporan
    print("\n" + "┌" + "─"*78 + "┐")
    print(f"│ {'REKAPITULASI HEAD-TO-HEAD BENCHMARK EVALUATION'.center(76)} │")
    print("├" + "─"*78 + "┤")

    # Menggunakan format to_string bawaan dengan pembatas kustom
    print(df_report.to_string(index=False, justify='center', col_space=15))

    print("├" + "─"*78 + "┤")
    print(f"│ 🏆 Rata-rata NDCG Legal Tina    : {mean_tina:.4f}".ljust(77) + "│")
    print(f"│ 🏆 Rata-rata NDCG Vector Search : {mean_vector:.4f}".ljust(77) + "│")
    print("└" + "─"*78 + "┘\n")

### 🎬 Eksekusi Perbandingan
Pastikan nama file CSV di bawah ini sesuai dengan file Ground Truth terbaru tempat kamu merekam data peringkat kedua sistem.

In [19]:
# Masukkan nama file Ground Truth Compare milikmu di sini
GT_FILE = "assets/Ground truth - compare.csv"
evaluate_compare_gt(GT_FILE, limit_k=5)

📖 Membaca file Ground Truth Compare: assets/Ground truth - compare.csv

🏆 HASIL ADU PERFORMA SISTEM: LEGAL TINA VS VECTOR SEARCH (NDCG@5)

▶️ [QUERY Q01] 'rules of origin'
   Legal Tina    -> Urutan Skor Relevansi: [0, 0, 2, 2, 2] | NDCG: 0.3821
   Vector Search -> Urutan Skor Relevansi: [2, 2, 1, 3, 2] | NDCG: 0.8450

▶️ [QUERY Q02] 'certificate of origin'
   Legal Tina    -> Urutan Skor Relevansi: [0, 3, 1, 1, 2] | NDCG: 0.4067
   Vector Search -> Urutan Skor Relevansi: [0, 3, 3, 3, 3] | NDCG: 0.6608

▶️ [QUERY Q03] 'dispute settlement'
   Legal Tina    -> Urutan Skor Relevansi: [0, 0, 2, 2, 2] | NDCG: 0.4469
   Vector Search -> Urutan Skor Relevansi: [2, 2, 2, 1, 1] | NDCG: 0.8614

┌──────────────────────────────────────────────────────────────────────────────┐
│                REKAPITULASI HEAD-TO-HEAD BENCHMARK EVALUATION                │
├──────────────────────────────────────────────────────────────────────────────┤
    Query ID            Query          NDCG Legal Tina  NDCG Ve